# 02．通道模型：從距離到掉包結果

- 本版已完成逐段導讀與人工接受。
- 核心程式位於 `src/channel/`。
- 本 Notebook 匯入同一份核心，保留說明與示範。
- 先看公式，再看程式，最後核對情境結果。
- **文獻公式**：說明通訊模型的計算依據。
- **研究假設**：說明本實驗選擇的接收規則。
- **測試設定**：說明目前選用的數值。
- 這三類資料不能互相代替。

|輸入|本版處理|輸出|
|---|---|---|
|既有拓樸、窗 ID、CW 檔位、k/n|往返路損、窗內偏移、有限碼長近似|接收品質、掉包率與逐包結果|

- CW 是供 Tag 反射的連續載波。
- SNR 是訊號功率除以噪聲功率。
- SINR 的分母另含外部干擾。
- 有限碼長近似估計短封包的掉包率。
- 程式假設封包已具備傳送條件。
- 能量、接入及時間檢查留給後續模組。

**來源**：[已接受配置](../Plan/channel-entry-pack.md)、[模型決策 §33](../ReferDoc/Plan/R-01_System_Model_and_Baseline_Decision_Record.md)。

## 1．載入既有拓樸與計算工具

- `iab` 提供已接受的拓樸資料。
- `dataclass(frozen=True)` 禁止欄位重新賦值。
- NumPy 處理穩定的對數運算。
- SciPy 計算積分與常態分布機率。
- 本 cell 不產生任何通道亂數。
- 路徑可從專案根目錄或 Notebook 目錄執行。

**來源**：[Python dataclasses](https://docs.python.org/3/library/dataclasses.html)、[SciPy quad](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html)、[既有拓樸程式](../src/iab/topology.py)。工具文件解釋程式用法，不是物理模型依據。

In [1]:
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
import hashlib # handle safe hash message
import json
import math
import random
import sys
import warnings

import numpy as np
from scipy.integrate import quad, IntegrationWarning
from scipy.special import log_ndtr

# back the last file
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Prevent the move file
# ROOT.parent like '' bash notebook ..''
if not (ROOT / 'src/iab').is_dir():
    raise RuntimeError('請從專案根目錄或 notebooks 目錄執行')

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from iab.contracts import TagZone, TopologySnapshot, TagRecord, Position2D
from iab.topology import build_topology, example_config, classify_distance, validate_config

## 2．集中管理設定

- 設定與資料格式集中於 `src/channel/contracts.py`。
- 集合只接受 `tuple`，不自動轉換 `list`。
- `profile_id` 綁定這組測試參數。
- 本版只接受已核對的 v0.1 配置。
- `channel_seed` 控制窗內通道偏移。
- `phy_seed` 控制封包接收抽樣。
- 兩個 seed 不同，避免混用亂數。

|欄位|意義|單位／測試值|
|---|---|---|
|frequency_hz|參考頻率|915 MHz|
|reference_gain_linear|1 m 的單程功率增益|無單位，約0.000679797|
|path_loss_exponent|單程衰減指數|無單位，2|
|reflection_power_coefficient|反射功率係數 beta|無單位，0.1|
|cw_power_dbm|低、中、高 CW 功率|dBm，-2／10／22|
|noise_power_dbm|接收器噪聲功率|dBm，-128|
|interference_dbm|各 Node 的固定外部干擾|dBm，每個 -128|
|bandwidth_hz|等效噪聲頻寬 B|Hz，10000|
|symbol_rate_hz|有效 BPSK 使用速率 Rs|每秒使用次數，10000|

- 噪聲測試值：`-174 + 10log10(10000) + 6 = -128 dBm`。
- `-174 dBm/Hz` 與噪聲指數 `6 dB` 為本次設定。
- 自身殘餘干擾為 **0 W**，不等於 0 dBm。
- 本 cell 不把這些值宣稱為戶外量測結果。

**來源**：[entry pack §2](../Plan/channel-entry-pack.md#2-候選測試參數)、[機讀配置](../Plan/channel-test-profile.json)、[研究決策 §33](../ReferDoc/Plan/R-01_System_Model_and_Baseline_Decision_Record.md)。915 MHz、beta、噪聲與 CW 檔位都屬測試配置。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [2]:
from channel.contracts import PowerLevel, ChannelConfig
from channel.channel_model import (finite_number, validate_channel_config,
                                   require_positive_int, require_id)


## 3．由距離計算接收品質

- `d` 是 Tag 到所屬 Node 的距離，單位 m。
- `gbar` 是單程功率增益，沒有單位。
- `delta` 是整條反射鏈路的偏移，單位 dB。
- `a = 10^(delta/10)`，只乘一次。

$$\bar g(d)=g_0(d/d_0)^{-\eta}$$
$$P_{rx}=P_{CW}\,\beta\,\bar g(d)^2\,10^{\delta/10}$$
$$\mathrm{SNR}_{RF}=P_{rx}/N,\qquad\mathrm{SINR}_{RF}=P_{rx}/(N+I)$$

- 功率公式中的功率都用 W。
- 程式在 dB 中相加，最後再轉回 W。
- `N + I` 必須先轉成 W 再相加。
- 本版將外部干擾近似成高斯噪聲。
- 同 Node 同窗使用相同偏移及接收器設定。
- 因此距離越近，實算接收品質越好。
- `[1, 50] m` 以外就拒絕，不移動 Tag。

**文獻公式**：[ITU-R P.525-5 §2.3，式(4)–(5)](https://www.itu.int/dms_pubrec/itu-r/rec/p/R-REC-P.525-5-202411-I!!PDF-E.pdf) 支持自由空間單程參考增益；[Al-Nahari 等，IEEE TCCN，§II-A 式(1)–(3)](https://doi.org/10.1109/TCCN.2025.3586219) 提供共址背散射的雙程模型。

**研究假設／測試設定**：[entry pack](../Plan/channel-entry-pack.md)。共同偏移、固定外干擾與有效距離由本研究指定，不是上述文獻的通用規定。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [3]:
from channel.contracts import LinkBudget
from channel.channel_model import link_budget


## 4．估計短封包的掉包率

- 本研究先使用理想同調 BPSK 接收器。
- 同調表示接收器已知目前相位。
- 本版也假設接收器已知目前通道品質。
- `X` 只取 `-1` 或 `+1`。
- `Z` 是平均 0、變異數 1 的高斯噪聲。

$$Y=\sqrt{\rho}X+Z,\qquad\rho=2\,\mathrm{SINR}_{RF}\,B/R_s$$

- `rho` 是實數 BPSK 模型的線性品質值。
- 係數 2 對應實數噪聲分量的 `N0/2`。
- 此處的 `B/Rs = 1`。
- 本版尚未映射標準中的實際符號格式。

$$i(Z)=1-\log_2(1+e^{-2\rho-2\sqrt{\rho}Z})$$
$$C=E[i(Z)],\qquad V=E[(i(Z)-C)^2]$$
$$\epsilon_{NA}\approx Q\left(\frac{nC-k}{\sqrt{nV}}\right)$$

|符號|意義|單位|
|---|---|---|
|k|資訊位元數|bit|
|n|可用 BPSK 次數|channel uses|
|C|每次使用的容量|bit/use|
|V|資訊密度的變異數|bit²/use|
|epsilon_NA|normal approximation 掉包率|0到1|

- 這是編碼理論近似，不是特定實作碼的保證。
- 原文採自然對數，程式除以 `ln(2)` 轉成 bit。
- 程式未加入更高階修正項。
- 浮點機率變成 0 或 1 時，保留對數機率。
- 求積無法解析變異數時，回報不可用。
- 對數機率超界時，也回報不可用。
- `log_error` 與 `log_success` 使用自然對數。
- 本版以 `V <= 1e-12` 作數值診斷界線。
- `rho > 64` 時也停止近似計算。
- 這些界線是求積保護，不是通道門檻。

**文獻公式**：[Durisi／Lancho，第3章](https://gdurisi.github.io/fbl-notes/bi-awgn.html)，§3.1 式(3.1)、§3.2 式(3.5)–(3.6)、§3.5.1 式(3.20)、§3.5.2 式(3.25)。原始有限碼長基礎另見 [Polyanskiy、Poor、Verdú，IEEE TIT 2010](https://doi.org/10.1109/TIT.2010.2043769)。上式為忽略高階項的近似。

**測試設定**：使用 `k=64,n=128`。另測 `32/128` 與 `64/256`，見 [12筆獨立參考值](../Plan/channel-fbl-reference.csv)。這不是 [TS 38.291 §6.1.2、§8.2、§8.4](../ReferDoc/Spec/A_IoT/TS38291_PHY.pdf) 的正式 TBS／CRC／編碼配置。

**程式工具**：[SciPy log_ndtr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.log_ndtr.html) 用於保存常態分布的對數尾端機率。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [4]:
from channel.contracts import FBLReference
from channel.channel_model import fbl_reference


## 5．按功率表選擇接收規則

- 本研究先選接收類別，再選掉包率。
- 實算 SNR／SINR 不會被表格改寫。
- `fbl` 保存公式估計的原始結果。
- `used_error_probability` 保存實際採用的掉包率。
- 兩者可以不同，`rule_reason` 會說明原因。

|CW 檔位|近區|中區|遠區|
|---|---|---|---|
|LOW|可靠，掉包率0|隨機，使用FBL|不可達，掉包率1|
|MEDIUM|可靠，掉包率0|可靠，掉包率0|隨機，使用FBL|
|HIGH|可靠，掉包率0|可靠，掉包率0|可靠，掉包率0|

- LOW 的中區即使品質變好，也仍是隨機類。
- 必須把窗設成 MEDIUM，才改為可靠類。
- 可靠與不可達是研究假設，不是文獻定理。
- 隨機類若無法求積，程式會保留診斷。
- 程式不會替它捏造掉包率。

**來源層級**：[研究決策 §33.14](../ReferDoc/Plan/R-01_System_Model_and_Baseline_Decision_Record.md)、[功率與鏈路綁定](../Plan/channel-link-binding.md)。本表是使用者接受的規則，不出自 3GPP／IEEE。隨機類公式依據見上一 cell 的 [Durisi 第3章](https://gdurisi.github.io/fbl-notes/bi-awgn.html)。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [5]:
from channel.contracts import LinkResult
from channel.channel_model import evaluate_link


## 6．將掉包率轉成單次封包結果

- 同窗的接收品質固定。
- 不同封包仍可有不同結果。
- 本版假設封包噪聲在給定通道後獨立。
- SHA256 將 seed 與傳輸身分綁在一起。
- 同一傳輸身分會得到同一個抽樣值。
- 所有亂數只在局部產生。
- 程式不改動 Python 全域亂數狀態。

- `uniform_draw` 是 0 與 1 之間的抽樣值。
- 程式使用 52 bit 的中點格點。
- 最小抽樣值是 `2^-53`。
- 若少數事件機率更小，會標示解析度限制。
- 程式保留 RANDOM 類別及對數機率。
- 少量抽樣不能驗證極小機率的準確度。

**來源（研究假設）**：[entry pack 的亂數與尾端處理](../Plan/channel-entry-pack.md)、[研究決策 §33](../ReferDoc/Plan/R-01_System_Model_and_Baseline_Decision_Record.md)。條件式逐包獨立與有限格點抽樣是本版模擬方法。

**程式工具**：[Python random](https://docs.python.org/3/library/random.html)、[hashlib](https://docs.python.org/3/library/hashlib.html)。SHA256 用於穩定重現身分，不用來聲稱射頻噪聲已驗證。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [6]:
from channel.contracts import ReceptionResult
from channel.channel_model import local_rng, draw_reception


## 7．管理同窗固定與跨窗改變

- 呼叫者明確開啟一個窗。
- 窗 ID 只表示順序，不產生時間軸。
- 本版不假設每隔 600 ms 都有一個新窗。
- 每個 Node 在每窗抽一次共同偏移。
- 範圍為 `[-3, +3] dB`，使用均勻分布。
- 各 Node 的偏移獨立產生。
- `window` 重讀當前窗時，不會重抽。
- 同窗不能改 CW 檔位。
- `link` 只能讀已開啟的當前窗。
- `receive` 對同一傳輸保留同一結果。
- 錯誤輸入不會更新目前的窗。
- 跨窗數值碰撞會報錯，不偷偷調整偏移。

|公開呼叫|作用|
|---|---|
|ChannelModel(config, topology)|檢查並綁定設定與拓樸|
|window(0, PowerLevel.LOW)|開啟低功率窗|
|link(0, 'Tag-001')|讀取品質與掉包率|
|receive(0, 'Tag-001', 'packet-1')|取得這次傳輸的結果|

**來源（研究假設）**：[模型草案的分窗規則](../Plan/channel-model-draft.md)、[已接受 entry pack](../Plan/channel-entry-pack.md)。同窗固定、共同偏移及功率綁定都由本研究指定。這不是從 3GPP 推導的 600 ms 通道相干時間。

- 核心已移至 [channel_model.py](../src/channel/channel_model.py)。
- 資料格式集中於 [contracts.py](../src/channel/contracts.py)。


In [7]:
from channel.contracts import WindowSnapshot
from channel.channel_model import ChannelModel


## 8．先看一條 20 m 鏈路

- 此例把偏移設為 0，方便手算。
- 這是公式檢查，不會改動實際窗的亂數。
- LOW 實際功率為 `-2 dBm`。
- 預期 SINR 約 `-2.404 dB`。
- 預期掉包率約 `0.327073`。
- 表中會保留原始與採用機率。

**來源**：[已接受配置](../Plan/channel-entry-pack.md)、[9格獨立參考值](../Plan/channel-link-reference.csv)。接收功率依第3節文獻公式；掉包率依第4節有限碼長近似。這是計算示例，不是量測結果。

In [8]:
from IPython.display import display, Markdown


def show_table(rows):
    headers = tuple(rows[0])
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join('---' for _ in headers) + ' |']
    for row in rows:
        values = [f'{row[h]:.6g}' if isinstance(row[h], float) else str(row[h]) for h in headers]
        lines.append('| ' + ' | '.join(values) + ' |')
    display(Markdown('\n'.join(lines)))


config = ChannelConfig()
worked = evaluate_link(config, 20.0, PowerLevel.LOW, 0.0, 0, window_id=0,
                       node_id='Node-1', tag_id='example-tag', zone=TagZone.MID)
show_table([{'CW (dBm)': worked.budget.cw_power_dbm,
             '接收功率 (dBm)': worked.budget.received_dbm,
             'SINR (dB)': worked.budget.sinr_db,
             'rho (線性)': worked.budget.rho,
             '接收類別': worked.receive_class,
             '原始掉包率': worked.fbl.epsilon,
             '採用掉包率': worked.used_error_probability}])

| CW (dBm) | 接收功率 (dBm) | SINR (dB) | rho (線性) | 接收類別 | 原始掉包率 | 採用掉包率 |
| --- | --- | --- | --- | --- | --- | --- |
| -2 | -127.394 | -2.40391 | 1.14984 | RANDOM | 0.327073 | 0.327073 |

## 9．核對三種功率與三個距離

- 近區使用 5 m。
- 中區使用 20 m。
- 遠區使用 40 m。
- 每格都用相同 `k=64,n=128`。
- `None` 表示此處無法提供原始近似值。
- 它不影響表格明定的可靠／不可達類別。
- 表格會顯示原因與數值狀態。

**來源**：[研究功率表與綁定](../Plan/channel-link-binding.md)、[9格參考表](../Plan/channel-link-reference.csv)。接收類別來自研究假設；原始機率來自 [Durisi §3.5](https://gdurisi.github.io/fbl-notes/bi-awgn.html)。

In [9]:
representative = []
for power in PowerLevel:
    for zone, distance in zip(TagZone, (5.0, 20.0, 40.0)):
        link = evaluate_link(config, distance, power, 0.0, 0, window_id=0,
                             node_id='Node-1', tag_id='example-tag', zone=zone)
        representative.append({'功率': power.value, '區域': zone.value,
                               'SINR (dB)': link.budget.sinr_db,
                               '接收類別': link.receive_class,
                               '原始掉包率': link.fbl.epsilon,
                               '採用掉包率': link.used_error_probability,
                               '數值狀態': link.fbl.status})
show_table(representative)

| 功率 | 區域 | SINR (dB) | 接收類別 | 原始掉包率 | 採用掉包率 | 數值狀態 |
| --- | --- | --- | --- | --- | --- | --- |
| LOW | NEAR | 21.6785 | RELIABLE | None | 0 | QUADRATURE_PRECISION_LIMIT |
| LOW | MID | -2.40391 | RANDOM | 0.327073 | 0.327073 | FINITE_REFERENCE |
| LOW | FAR | -14.4451 | UNREACHABLE | 1 | 1 | FLOAT_TAIL |
| MEDIUM | NEAR | 33.6785 | RELIABLE | None | 0 | QUADRATURE_PRECISION_LIMIT |
| MEDIUM | MID | 9.59609 | RELIABLE | 0 | 0 | FLOAT_TAIL |
| MEDIUM | FAR | -2.44511 | RANDOM | 0.343336 | 0.343336 | FINITE_REFERENCE |
| HIGH | NEAR | 45.6785 | RELIABLE | None | 0 | QUADRATURE_PRECISION_LIMIT |
| HIGH | MID | 21.5961 | RELIABLE | None | 0 | QUADRATURE_PRECISION_LIMIT |
| HIGH | FAR | 9.55489 | RELIABLE | 0 | 0 | FLOAT_TAIL |

## 10．套用已接受的 30 個 Tag

- 本 cell 讀取既有拓樸的 `snapshot.json`。
- JSON 的陣列轉回既有 DTO 的 tuple。
- 載入步驟不產生任何新位置。
- 通道模型會檢查距離與所屬 Node。
- 本 cell 先讀兩次低功率窗 0。
- 然後開啟中功率窗 1。
- 表格顯示每個 Node 的共同偏移。
- 同窗重讀資料相同；跨窗偏移不同。
- SNR 變化另含 CW 檔位的差異。

**來源**：[已接受拓樸快照](artifacts/iab-topology/snapshot.json)、[IAB DTO](../src/iab/contracts.py)、[分窗假設與測試配置](../Plan/channel-entry-pack.md)。JSON 還原 tuple 是檔案載入，不是放寬設定輸入規則。

In [10]:
from iab.contracts import FixedNodeConfig, TopologyConfig, UERecord

payload = json.loads((ROOT / 'notebooks/artifacts/iab-topology/snapshot.json').read_text())
raw = payload['config']


def station_from_json(station):
    return FixedNodeConfig(station['node_id'], Position2D(**station['position']))


topology_config = TopologyConfig(
    donor=station_from_json(raw['donor']),
    iab_nodes=tuple(station_from_json(node) for node in raw['iab_nodes']),
    zone_outer_radii_m=tuple(raw['zone_outer_radii_m']),
    zone_tag_counts=tuple(raw['zone_tag_counts']),
    min_tags_per_node=raw['min_tags_per_node'], seed=raw['seed'])

topology = TopologySnapshot(
    topology_config,
    tuple(UERecord(**ue) for ue in payload['ues']),
    tuple(TagRecord(tag['tag_id'], tag['parent_node_id'], TagZone(tag['zone']),
                    Position2D(**tag['position'])) for tag in payload['tags']))

model = ChannelModel(config, topology)
window0 = model.window(0, PowerLevel.LOW)
links0 = tuple(model.link(0, tag.tag_id) for tag in topology.tags)

assert model.window(0, PowerLevel.LOW) == window0
assert tuple(model.link(0, tag.tag_id) for tag in topology.tags) == links0
window1 = model.window(1, PowerLevel.MEDIUM)

links1 = tuple(model.link(1, tag.tag_id) for tag in topology.tags)

show_table([{'Node': node_id, '窗0偏移 (dB)': delta,
             '窗1偏移 (dB)': dict(window1.node_offsets_db)[node_id]}
            for node_id, delta in window0.node_offsets_db])

print(f'已載入 {len(topology.tags)} 個 Tag；同窗重讀相同；目前為中功率窗1。')

| Node | 窗0偏移 (dB) | 窗1偏移 (dB) |
| --- | --- | --- |
| Node-1 | 0.0682785 | -2.64489 |
| Node-2 | 1.24578 | 1.08159 |
| Node-3 | -0.224881 | 1.67738 |
| Node-4 | -0.839946 | 0.915911 |
| Node-5 | 2.80177 | -2.00344 |

已載入 30 個 Tag；同窗重讀相同；目前為中功率窗1。


## 11．查看封包結果與浮點尾端

- 表格列出窗 1 的 30 個 Tag。
- 每個 Tag 使用一個明確的傳輸 ID。
- 再讀一次同傳輸，結果不會重抽。
- `失敗` 只代表已傳送封包的接收結果。
- `FINITE_PRNG_RESOLUTION_LIMIT` 表示亂數格點不足。
- 即使顯示 0 或 1，RANDOM 仍保留原類別。
- 完整資料含 profile、seed、k/n 及接收器設定。
- 下方另顯示一筆完整資料，方便追查。
- 最後用既有 RANDOM 尾端算例核對。
- 該算例不會更新目前窗。

**來源**：[功率與鏈路綁定](../Plan/channel-link-binding.md)、[entry pack 的尾端限制](../Plan/channel-entry-pack.md)、[既有尾端算例](../Plan/channel-random-region-scan.csv)。逐包獨立為研究假設；FBL 公式見 [Durisi 第3章](https://gdurisi.github.io/fbl-notes/bi-awgn.html)。有限亂數無法證明極小掉包率。

In [11]:
packets = tuple(model.receive(1, tag.tag_id, f'window1-{tag.tag_id}') for tag in topology.tags)
assert packets[0] == model.receive(1, topology.tags[0].tag_id, f'window1-{topology.tags[0].tag_id}')

show_table([{'Tag': p.link.tag_id, 'Node': p.link.node_id, '區域': p.link.zone.value,
             'SINR (dB)': p.link.budget.sinr_db, '接收類別': p.link.receive_class,
             '掉包率': p.link.used_error_probability, '失敗': p.failed,
             '抽樣狀態': p.sampling_status} for p in packets])
display(Markdown('**一筆完整接收紀錄**'))

print(json.dumps(asdict(packets[0]), ensure_ascii=False, indent=2, allow_nan=False))

import csv

with (ROOT / 'Plan/channel-random-region-scan.csv').open(encoding='utf-8-sig') as stream:
    tail_case = next(row for row in csv.DictReader(stream)
                     if row['tag_id'] == 'Tag-008' and float(row['offset_db']) == 3.0)
    
tail_tag = next(tag for tag in topology.tags if tag.tag_id == tail_case['tag_id'])
tail_node_index = next(i for i, node in enumerate(topology.config.iab_nodes)
                       if node.node_id == tail_tag.parent_node_id)

tail_link = evaluate_link(config, float(tail_case['distance_m']), PowerLevel.LOW,
                          3.0, tail_node_index, window_id=0, node_id=tail_tag.parent_node_id,
                          tag_id=tail_tag.tag_id, zone=tail_tag.zone)

tail_packet = draw_reception(config, tail_link, 'tail-example')

display(Markdown('**RANDOM 尾端算例：顯示為0，仍保留對數機率**'))
show_table([{'Tag': tail_link.tag_id, '接收類別': tail_link.receive_class,
             '顯示掉包率': tail_link.fbl.epsilon, 'log(error)': tail_link.fbl.log_error,
             '數值狀態': tail_link.fbl.status, '抽樣狀態': tail_packet.sampling_status}])


| Tag | Node | 區域 | SINR (dB) | 接收類別 | 掉包率 | 失敗 | 抽樣狀態 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Tag-001 | Node-5 | MID | 6.80202 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-002 | Node-4 | MID | 3.82956 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-003 | Node-3 | MID | 6.81278 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-004 | Node-2 | FAR | -3.83865 | RANDOM | 0.850794 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-005 | Node-2 | MID | 6.09914 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-006 | Node-1 | NEAR | 22.9346 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-007 | Node-4 | NEAR | 24.5952 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-008 | Node-1 | MID | 15.4315 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-009 | Node-3 | MID | 11.0021 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-010 | Node-5 | MID | 9.62093 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-011 | Node-2 | NEAR | 26.8634 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-012 | Node-3 | FAR | -0.0670556 | RANDOM | 0.000429193 | False | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-013 | Node-1 | FAR | -3.95876 | RANDOM | 0.876936 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-014 | Node-1 | MID | 10.0741 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-015 | Node-4 | MID | 3.71979 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-016 | Node-1 | FAR | -7.33378 | RANDOM | 0.999997 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-017 | Node-3 | NEAR | 48.1834 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-018 | Node-3 | MID | 7.07972 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-019 | Node-5 | MID | 4.31312 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-020 | Node-3 | NEAR | 28.6484 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-021 | Node-1 | MID | 2.05702 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-022 | Node-4 | FAR | -3.80645 | RANDOM | 0.843176 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-023 | Node-5 | NEAR | 39.4468 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-024 | Node-1 | FAR | -7.65268 | RANDOM | 0.999999 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-025 | Node-4 | NEAR | 28.7409 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-026 | Node-1 | NEAR | 19.4116 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-027 | Node-5 | FAR | -3.86528 | RANDOM | 0.856898 | True | CONDITIONAL_BERNOULLI_52_BIT_GRID |
| Tag-028 | Node-3 | NEAR | 28.0785 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-029 | Node-5 | NEAR | 23.8087 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |
| Tag-030 | Node-3 | MID | 9.74962 | RELIABLE | 0 | False | DETERMINISTIC_RESEARCH_RULE |

**一筆完整接收紀錄**

{
  "transmission_id": "window1-Tag-001",
  "link": {
    "window_id": 1,
    "channel_seed": 20261029,
    "node_id": "Node-5",
    "tag_id": "Tag-001",
    "zone": "MID",
    "budget": {
      "profile_id": "channel-entry-synthetic-v0.1",
      "power_level": "MEDIUM",
      "distance_m": 20.931275254857027,
      "offset_db": -2.0034393871971847,
      "cw_power_dbm": 10.0,
      "cw_power_w": 0.01,
      "received_dbm": -118.18767714910076,
      "received_w": 1.5178619882627682e-15,
      "noise_w": 1.5848931924611109e-16,
      "interference_w": 1.5848931924611109e-16,
      "residual_self_interference_w": 0.0,
      "snr_db": 9.812322850899253,
      "sinr_db": 6.802022894259441,
      "rho": 9.57706169401704
    },
    "fbl": {
      "model_id": "bi-awgn-normal-approximation-no-higher-order-v1",
      "rho": 9.57706169401704,
      "k": 64,
      "n": 128,
      "capacity_bits": 0.9959295499804073,
      "dispersion_bits2": 0.012853168349628687,
      "epsilon": 0.0,
      "log

**RANDOM 尾端算例：顯示為0，仍保留對數機率**

| Tag | 接收類別 | 顯示掉包率 | log(error) | 數值狀態 | 抽樣狀態 |
| --- | --- | --- | --- | --- | --- |
| Tag-008 | RANDOM | 0 | -37907.9 | FLOAT_TAIL | FINITE_PRNG_RESOLUTION_LIMIT |

## 12．固定品質，比較資料量與碼長

- 本例固定 `rho=1`。
- `32/128` 的資料量較少。
- `64/256` 允許更多通道使用次數。
- 本例可觀察兩者如何影響掉包率。
- 這裡不加上接入、重傳或等待時間。

**文獻公式**：[Durisi §3.1、§3.5.2](https://gdurisi.github.io/fbl-notes/bi-awgn.html)。本例依相同品質比較 `k/n`；已核對的數值來源為 [12筆參考表](../Plan/channel-fbl-reference.csv)。

In [12]:
show_table([{'k (bit)': k, 'n (uses)': n, 'rho': 1.0,
             '掉包率近似': fbl_reference(1.0, k, n).epsilon}
            for k, n in ((32, 128), (64, 128), (64, 256))])

| k (bit) | n (uses) | rho | 掉包率近似 |
| --- | --- | --- | --- |
| 32 | 128 | 1 | 0.000507018 |
| 64 | 128 | 1 | 0.577614 |
| 64 | 256 | 1 | 1.67618e-06 |

## 13．試一次超界輸入

- 本例輸入 `0.999 m`。
- 預期程式拒絕並顯示原因。
- 程式不把距離夾到 `1 m`。
- 原拓樸也不會被改動。

**來源**：[已接受的有效距離處理](../Plan/channel-entry-pack.md)、[研究決策 §33.15](../ReferDoc/Plan/R-01_System_Model_and_Baseline_Decision_Record.md)。拒絕行為是本研究選擇，不是文獻給定的傳播定律。

In [13]:
try:
    link_budget(config, 0.999, PowerLevel.LOW, 0.0, 0)
except ValueError as error:
    print('預期拒絕：', error)
else:
    raise AssertionError('超界輸入不應通過')

預期拒絕： distance must be within [1, 50] m; Tag is not moved


## 14．人工導讀與接受

- 先確認第2節的設定與單位。
- 再用第3、4節核對公式。
- 用第5節確認功率表的研究假設。
- 用第7節確認窗與傳輸 ID 的用途。
- 最後查看第8至13節的實際結果。
- 您已完成理解與接受；核心已抽離為 `.py`。

**本版限制**：測試值未經戶外量測校準。有限碼長是理論近似。極小機率不以少量封包抽樣證明。正式 TBS、能量及接入仍待後續模組整合。

**來源索引**：[產品操作與閱讀指南](../ReferDoc/Implementation/ChannelModel/README.md)、[配置與數值依據](../Plan/channel-entry-pack.md)。